![image.png](https://i.imgur.com/a3uAqnb.png)

# N-Gram Probabilistic Language Models - Homework Assignment

In this homework, you will implement **N-gram probabilistic language models** to analyze and generate text. This project will help you understand the fundamentals of statistical language modeling and text generation.

## 📌 Project Overview
- **Task**: Build Unigram, Bigram, and Trigram language models
- **Dataset**: Million Headlines dataset from News
- **Goal**: Understand probability distributions in text and implement text generation
- **Applications**: Text prediction, generation, and probability estimation

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand N-gram language models and their applications
- Learn text preprocessing and tokenization techniques
- Implement probability calculations and smoothing
- Build vector space models and similarity measures
- Practice text generation using probabilistic models
- Analyze linguistic patterns in real news data

## 1️⃣ Initial Setup and Dataset Loading

**Task**: Download the dataset and explore its basic structure.

**Requirements**:
- Download the Million Headlines dataset
- Load and examine the data structure
- Display basic dataset information

In [47]:
import kagglehub
import pandas as pd
import numpy as np
import re
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

path = kagglehub.dataset_download("therohk/million-headlines")
df = pd.read_csv(path + "/abcnews-date-text.csv")

print(f"Dataset loaded: {df.shape[0]:,} headlines")
print(f"Columns: {list(df.columns)}")
print(f"Sample headlines:")
for i in range(3):
    print(f"  {i+1}. {df.iloc[i]['headline_text']}")
print(f"Date range: {df['publish_date'].min()} to {df['publish_date'].max()}")

Dataset loaded: 1,244,184 headlines
Columns: ['publish_date', 'headline_text']
Sample headlines:
  1. aba decides against community broadcasting licence
  2. act fire witnesses must be aware of defamation
  3. a g calls for infrastructure protection summit
Date range: 20030219 to 20211231


## 2️⃣ Text Preprocessing and Tokenization

**Task**: Clean and tokenize the text data for language modeling.

**Requirements**:
- Implement text preprocessing function for English text
- Create tokenization function
- Add sentence boundary markers
- Process all headlines and create a unified corpus

In [48]:
def preprocess_english_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())

    return text.strip()

def tokenize_text(text):
    if not text:
        return []
    return text.split()

def add_sentence_boundaries(tokens):
    return ['<s>'] + tokens + ['</s>']

sample_df = df.copy()

sample_df['cleaned_text'] = sample_df['headline_text'].apply(preprocess_english_text)
sample_df['tokens'] = sample_df['cleaned_text'].apply(tokenize_text)
sample_df['tokens_with_boundaries'] = sample_df['tokens'].apply(add_sentence_boundaries)

sample_df = sample_df[sample_df['tokens'].apply(len) > 0]

print(f"Processed {len(sample_df)} headlines")
print(f"Sample tokens: {sample_df.iloc[0]['tokens']}")
print(f"With boundaries: {sample_df.iloc[0]['tokens_with_boundaries']}")

all_tokens = []
for tokens in sample_df['tokens_with_boundaries']:
    all_tokens.extend(tokens)

print(f"Total tokens in corpus: {len(all_tokens):,}")

vocab_counts = Counter(all_tokens)
print(f"Vocabulary size: {len(vocab_counts):,}")
print(f"Most common words: {vocab_counts.most_common(10)}")

Processed 1244182 headlines
Sample tokens: ['aba', 'decides', 'against', 'community', 'broadcasting', 'licence']
With boundaries: ['<s>', 'aba', 'decides', 'against', 'community', 'broadcasting', 'licence', '</s>']
Total tokens in corpus: 10,572,842
Vocabulary size: 101,960
Most common words: [('<s>', 1244182), ('</s>', 1244182), ('to', 238380), ('in', 156206), ('for', 143279), ('of', 95941), ('on', 82062), ('the', 65067), ('over', 54547), ('police', 39850)]


## 3️⃣ N-gram Model Implementation

**Task**: Implement N-gram language models with proper OOV handling.

**Requirements**:
- Create vocabulary with frequency filtering
- Handle out-of-vocabulary (OOV) words
- Implement N-gram model class with training and probability calculation
- Build Unigram, Bigram, and Trigram models

In [49]:
def create_vocabulary(token_counts, min_freq=2, max_vocab_size=10000):
    frequent_words = {word: count for word, count in token_counts.items() if count >= min_freq}
    sorted_words = sorted(frequent_words.items(), key=lambda x: x[1], reverse=True)
    vocab = {word for word, count in sorted_words[:max_vocab_size]}
    return vocab

def replace_oov_words(tokens, vocabulary):
    return [token if token in vocabulary else '<UNK>' for token in tokens]

vocabulary = create_vocabulary(vocab_counts, min_freq=2, max_vocab_size=99999999999)
vocabulary.add('<UNK>')

print(f"Vocabulary size after filtering: {len(vocabulary)}")

processed_tokens = []
for tokens in sample_df['tokens_with_boundaries']:
    tokens_with_unk = replace_oov_words(tokens, vocabulary)
    processed_tokens.extend(tokens_with_unk)

print(f"Total processed tokens: {len(processed_tokens):,}")

unk_count = processed_tokens.count('<UNK>')
print(f"UNK tokens: {unk_count:,} ({unk_count/len(processed_tokens)*100:.1f}%)")

class NGramModel:
    def __init__(self, n):
        self.n = n
        self.counts = defaultdict(int)
        self.context_counts = defaultdict(int)

    def train(self, tokens):
        for i in range(len(tokens) - self.n + 1):
            ngram = tuple(tokens[i:i + self.n])
            context = ngram[:-1] if self.n > 1 else ()

            self.counts[ngram] += 1
            self.context_counts[context] += 1

    def get_probability(self, ngram):
        ngram = tuple(ngram)
        context = ngram[:-1] if self.n > 1 else ()

        if self.context_counts[context] == 0:
            return 0.0

        return self.counts[ngram] / self.context_counts[context]

    def get_count(self, ngram):
        return self.counts[tuple(ngram)]

print("Building N-gram models...")

unigram_model = NGramModel(1)
bigram_model = NGramModel(2)
trigram_model = NGramModel(3)

unigram_model.train(processed_tokens)
bigram_model.train(processed_tokens)
trigram_model.train(processed_tokens)

print(f"Unigram model: {len(unigram_model.counts):,} unique unigrams")
print(f"Bigram model: {len(bigram_model.counts):,} unique bigrams")
print(f"Trigram model: {len(trigram_model.counts):,} unique trigrams")

Vocabulary size after filtering: 62127
Total processed tokens: 10,572,842
UNK tokens: 39,834 (0.4%)
Building N-gram models...
Unigram model: 62,127 unique unigrams
Bigram model: 2,320,614 unique bigrams
Trigram model: 5,318,151 unique trigrams


## 4️⃣ Probability Analysis and Count/Probability Matrices

**Task**: Calculate probabilities and build count/probability matrices for analysis.

**Requirements**:
- Calculate example N-gram probabilities
- Build count matrices for visualization
- Convert count matrices to probability matrices
- Calculate sentence probabilities
- Find most probable next words given context

In [50]:
print("Example N-gram probabilities:")

words_to_test = ['police', 'government', 'school', 'fire']
for word in words_to_test:
    prob = unigram_model.get_probability([word])
    print(f"P({word}) = {prob:.6f}")

print("\nBigram probabilities:")
bigrams_to_test = [['police', 'arrest'], ['government', 'announces'], ['fire', 'burns'], ['school', 'students']]
for bigram in bigrams_to_test:
    prob = bigram_model.get_probability(bigram)
    print(f"P({bigram[1]}|{bigram[0]}) = {prob:.6f}")

print("\nTrigram probabilities:")
trigrams_to_test = [['police', 'arrest', 'man'], ['government', 'announces', 'new'], ['fire', 'burns', 'through']]
for trigram in trigrams_to_test:
    prob = trigram_model.get_probability(trigram)
    print(f"P({trigram[2]}|{trigram[0]} {trigram[1]}) = {prob:.6f}")

def build_count_matrix(model, vocab_subset):
    matrix = np.zeros((len(vocab_subset), len(vocab_subset)))
    vocab_to_idx = {word: i for i, word in enumerate(vocab_subset)}

    for ngram, count in model.counts.items():
        if len(ngram) == 2:
            word1, word2 = ngram
            if word1 in vocab_to_idx and word2 in vocab_to_idx:
                i, j = vocab_to_idx[word1], vocab_to_idx[word2]
                matrix[i][j] = count

    return matrix, vocab_to_idx

def build_probability_matrix(count_matrix, context_counts, vocab_subset, vocab_to_idx):
    prob_matrix = np.zeros_like(count_matrix)

    for i, word in enumerate(vocab_subset):
        context_count = context_counts.get((word,), 0)
        if context_count > 0:
            prob_matrix[i] = count_matrix[i] / context_count

    return prob_matrix

common_words = [word for word, count in vocab_counts.most_common(50)
                if word not in {'<s>', '</s>', 'to', 'in', 'for', 'of', 'on', 'the', 'a', 'and', 'with', 'at', 'from', 'by', 'as'}]
top_words = common_words[:15]

print(f"\nTop 15 content words for matrix: {top_words}")

count_matrix, vocab_to_idx = build_count_matrix(bigram_model, top_words)
print(f"Count matrix shape: {count_matrix.shape}")

prob_matrix = build_probability_matrix(count_matrix, bigram_model.context_counts, top_words, vocab_to_idx)

print(f"\nSample Count Matrix (first 6x6):")
print("Rows: previous word, Columns: next word")
print("      ", end="")
for j in range(6):
    print(f"{top_words[j]:>8}", end="")
print()
for i in range(6):
    print(f"{top_words[i]:>6}", end="")
    for j in range(6):
        print(f"{int(count_matrix[i][j]):>8}", end="")
    print()

print(f"\nSample Probability Matrix (first 6x6):")
print("      ", end="")
for j in range(6):
    print(f"{top_words[j]:>8}", end="")
print()
for i in range(6):
    print(f"{top_words[i]:>6}", end="")
    for j in range(6):
        print(f"{prob_matrix[i][j]:>8.4f}", end="")
    print()

def calculate_sentence_probability(sentence_tokens, model):
    if model.n == 1:
        prob = 1.0
        for token in sentence_tokens:
            prob *= model.get_probability([token])
        return prob

    elif model.n == 2:
        prob = 1.0
        for i in range(len(sentence_tokens) - 1):
            bigram_prob = model.get_probability([sentence_tokens[i], sentence_tokens[i+1]])
            prob *= bigram_prob
        return prob

    elif model.n == 3:
        prob = 1.0
        for i in range(len(sentence_tokens) - 2):
            trigram_prob = model.get_probability([sentence_tokens[i], sentence_tokens[i+1], sentence_tokens[i+2]])
            prob *= trigram_prob
        return prob

test_sentences = [
    ['<s>', 'police', 'arrest', 'man', '</s>'],
    ['<s>', 'government', 'announces', 'new', 'policy', '</s>'],
    ['<s>', 'school', 'fire', 'emergency', '</s>']
]

print(f"\nSentence Probability Examples:")
print("-" * 50)

for sentence in test_sentences:
    sentence_processed = replace_oov_words(sentence, vocabulary)

    unigram_prob = calculate_sentence_probability(sentence_processed, unigram_model)
    bigram_prob = calculate_sentence_probability(sentence_processed, bigram_model)
    trigram_prob = calculate_sentence_probability(sentence_processed, trigram_model)

    print(f"Sentence: {' '.join(sentence[1:-1])}")
    print(f"  Unigram P = {unigram_prob:.2e}")
    print(f"  Bigram P  = {bigram_prob:.2e}")
    print(f"  Trigram P = {trigram_prob:.2e}")
    print()

def get_most_probable_next_words(context, model, top_k=5):
    if model.n == 2:
        candidates = []

        for ngram, count in model.counts.items():
            if len(ngram) == 2 and ngram[0] == context:
                prob = model.get_probability(ngram)
                candidates.append((ngram[1], prob))

        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates[:top_k]

    return []

print("Most probable next words given context:")
print("-" * 40)

test_contexts = ['police', 'government', 'fire', 'school']
for context in test_contexts:
    if context in vocabulary:
        next_words = get_most_probable_next_words(context, bigram_model, top_k=5)
        print(f"After '{context}': {[(word, f'{prob:.3f}') for word, prob in next_words]}")

Example N-gram probabilities:
P(police) = 0.003769
P(government) = 0.000946
P(school) = 0.000775
P(fire) = 0.001469

Bigram probabilities:
P(arrest|police) = 0.019774
P(announces|government) = 0.017789
P(burns|fire) = 0.004184
P(students|school) = 0.022700

Trigram probabilities:
P(man|police arrest) = 0.161168
P(new|government announces) = 0.089888
P(through|fire burns) = 0.123077

Top 15 content words for matrix: ['over', 'police', 'after', 'new', 'man', 'says', 'up', 'us', 'out', 'be', 'court', 'australia', 'govt', 'council', 'more']
Count matrix shape: (15, 15)

Sample Count Matrix (first 6x6):
Rows: previous word, Columns: next word
          over  police   after     new     man    says
  over       6     317      38     320      21      14
police     117       0      76      12      16      32
 after       0     350       2      87     163       0
   new       1     177       4       5       5       2
   man     536      14     194       2       0      43
  says       9      84  

## 5️⃣ Vector Space Models and Similarity Measures

**Task**: Implement vector representations and similarity calculations.

**Requirements**:
- Create one-hot encoding vectors
- Build word co-occurrence matrix
- Implement similarity measures (Euclidean distance, cosine similarity)
- Normalize vectors and create dense embeddings
- Find similar words using vector representations

In [52]:
def create_one_hot_vectors(vocabulary):
    vocab_list = list(vocabulary)
    vocab_to_idx = {word: i for i, word in enumerate(vocab_list)}

    one_hot_vectors = {}
    for word in vocab_list:
        vector = np.zeros(len(vocab_list))
        vector[vocab_to_idx[word]] = 1
        one_hot_vectors[word] = vector

    return one_hot_vectors, vocab_to_idx

def create_cooccurrence_matrix(tokens, window_size=2, vocab_subset=None):
    if vocab_subset is None:
        vocab_subset = list(set(tokens))

    vocab_to_idx = {word: i for i, word in enumerate(vocab_subset)}
    matrix = np.zeros((len(vocab_subset), len(vocab_subset)))

    for i, center_word in enumerate(tokens):
        if center_word not in vocab_to_idx:
            continue

        center_idx = vocab_to_idx[center_word]

        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)

        for j in range(start, end):
            if i != j and tokens[j] in vocab_to_idx:
                context_idx = vocab_to_idx[tokens[j]]
                matrix[center_idx][context_idx] += 1

    return matrix, vocab_to_idx

def euclidean_distance(vec1, vec2):
    return np.sqrt(np.sum((vec1 - vec2) ** 2))

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)

print("Creating Vector Space Models...")

news_words = ['police', 'government', 'fire', 'school', 'court', 'hospital', 'council', 'water', 'health', 'road']
small_vocab = [word for word in news_words if word in vocabulary]

one_hot_vectors, vocab_to_idx = create_one_hot_vectors(small_vocab)

print(f"One-hot vectors created for {len(small_vocab)} words")
print(f"News words used: {small_vocab}")
print(f"Vector dimension: {len(one_hot_vectors[small_vocab[0]])}")

print(f"\nSample One-Hot Vectors:")
for i, word in enumerate(small_vocab[:5]):
    vector = one_hot_vectors[word]
    print(f"{word}: {vector}")

print(f"\nBuilding co-occurrence matrix...")
subset_tokens = processed_tokens
cooccurrence_matrix, cooc_vocab_to_idx = create_cooccurrence_matrix(
    subset_tokens,
    window_size=2,
    vocab_subset=small_vocab
)

print(f"Co-occurrence matrix shape: {cooccurrence_matrix.shape}")

def normalize_matrix(matrix, method='l2'):
    if method == 'l2':
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return matrix / norms
    elif method == 'sum':
        sums = np.sum(matrix, axis=1, keepdims=True)
        sums[sums == 0] = 1
        return matrix / sums
    return matrix

dense_embeddings = normalize_matrix(cooccurrence_matrix, method='l2')

print(f"Dense embeddings shape: {dense_embeddings.shape}")

print(f"\nCo-occurrence Matrix (first 5x5):")
print("       ", end="")
for j in range(5):
    print(f"{small_vocab[j]:>8}", end="")
print()
for i in range(5):
    print(f"{small_vocab[i]:>7}", end="")
    for j in range(5):
        print(f"{int(cooccurrence_matrix[i][j]):>8}", end="")
    print()

print(f"\nSimilarity Calculations:")
print("=" * 50)

test_word_pairs = [('police', 'court'), ('fire', 'hospital'), ('government', 'council'),
                   ('school', 'health'), ('water', 'road')]

valid_pairs = [(w1, w2) for w1, w2 in test_word_pairs
               if w1 in small_vocab and w2 in small_vocab]

print(f"Euclidean Distance (One-hot vectors):")
for word1, word2 in valid_pairs:
    if word1 in one_hot_vectors and word2 in one_hot_vectors:
        dist = euclidean_distance(one_hot_vectors[word1], one_hot_vectors[word2])
        print(f"  {word1:>10} - {word2:<10}: {dist:.3f}")

print(f"\nEuclidean Distance (Dense embeddings):")
for word1, word2 in valid_pairs:
    if word1 in cooc_vocab_to_idx and word2 in cooc_vocab_to_idx:
        idx1 = cooc_vocab_to_idx[word1]
        idx2 = cooc_vocab_to_idx[word2]
        dist = euclidean_distance(dense_embeddings[idx1], dense_embeddings[idx2])
        print(f"  {word1:>10} - {word2:<10}: {dist:.3f}")

print(f"\nCosine Similarity (Dense embeddings):")
for word1, word2 in valid_pairs:
    if word1 in cooc_vocab_to_idx and word2 in cooc_vocab_to_idx:
        idx1 = cooc_vocab_to_idx[word1]
        idx2 = cooc_vocab_to_idx[word2]
        sim = cosine_similarity(dense_embeddings[idx1], dense_embeddings[idx2])
        print(f"  {word1:>10} - {word2:<10}: {sim:.3f}")

def find_most_similar_words(target_word, embeddings, vocab_to_idx, top_k=3):
    if target_word not in vocab_to_idx:
        return []

    target_idx = vocab_to_idx[target_word]
    target_vector = embeddings[target_idx]

    similarities = []
    for word, idx in vocab_to_idx.items():
        if word != target_word:
            sim = cosine_similarity(target_vector, embeddings[idx])
            similarities.append((word, sim))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

print(f"\nMost similar words (using dense embeddings):")
for word in small_vocab[:5]:
    similar_words = find_most_similar_words(word, dense_embeddings, cooc_vocab_to_idx, top_k=3)
    print(f"'{word:>10}': {[(w, f'{s:.3f}') for w, s in similar_words]}")

print(f"\nVector Arithmetic Examples:")
word1, word2, word3 = small_vocab[0], small_vocab[1], small_vocab[2]
word1_idx = cooc_vocab_to_idx.get(word1)
word2_idx = cooc_vocab_to_idx.get(word2)
word3_idx = cooc_vocab_to_idx.get(word3)

if all(idx is not None for idx in [word1_idx, word2_idx, word3_idx]):
    result_vector = dense_embeddings[word1_idx] - dense_embeddings[word2_idx] + dense_embeddings[word3_idx]

    best_sim = -1
    best_word = None
    for word, idx in cooc_vocab_to_idx.items():
        sim = cosine_similarity(result_vector, dense_embeddings[idx])
        if sim > best_sim:
            best_sim = sim
            best_word = word

    print(f"{word1} - {word2} + {word3} ≈ {best_word} (similarity: {best_sim:.3f})")
else:
    print("Some words not found in co-occurrence matrix")

Creating Vector Space Models...
One-hot vectors created for 10 words
News words used: ['police', 'government', 'fire', 'school', 'court', 'hospital', 'council', 'water', 'health', 'road']
Vector dimension: 10

Sample One-Hot Vectors:
police: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
government: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
fire: [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
school: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
court: [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]

Building co-occurrence matrix...
Co-occurrence matrix shape: (10, 10)
Dense embeddings shape: (10, 10)

Co-occurrence Matrix (first 5x5):
         policegovernment    fire  school   court
 police      10      19     177      61     165
government      19       0      42      12      14
   fire     177      42      24     166      29
 school      61      12     166       6      23
  court     165      14      29      23       2

Similarity Calculations:
Euclidean Distance (One-hot vectors):
      police - court     : 1.414
        fire - hospital  : 1.414
  g

## 6️⃣ Text Generation with N-gram Models

**Task**: Implement text generation using trained N-gram models.

**Requirements**:
- Create text generator class
- Implement probabilistic sampling
- Generate text from scratch and with seed text
- Experiment with different temperature settings
- Compare deterministic vs random generation

In [53]:
import random

class TextGenerator:
    def __init__(self, ngram_model):
        self.model = ngram_model
        self.n = ngram_model.n

    def get_next_word_candidates(self, context):
        candidates = []

        if self.n == 1:
            for ngram, count in self.model.counts.items():
                word = ngram[0]
                prob = self.model.get_probability([word])
                if prob > 0:
                    candidates.append((word, prob))

        elif self.n == 2:
            if len(context) >= 1:
                last_word = context[-1]
                for ngram, count in self.model.counts.items():
                    if len(ngram) == 2 and ngram[0] == last_word:
                        next_word = ngram[1]
                        prob = self.model.get_probability(ngram)
                        if prob > 0:
                            candidates.append((next_word, prob))

        elif self.n == 3:
            if len(context) >= 2:
                context_tuple = tuple(context[-2:])
                for ngram, count in self.model.counts.items():
                    if len(ngram) == 3 and ngram[:2] == context_tuple:
                        next_word = ngram[2]
                        prob = self.model.get_probability(ngram)
                        if prob > 0:
                            candidates.append((next_word, prob))

        return candidates

    def sample_next_word(self, context, temperature=1.0):
        candidates = self.get_next_word_candidates(context)

        if not candidates:
            return None

        words, probs = zip(*candidates)
        probs = np.array(probs)

        if temperature != 1.0:
            probs = np.power(probs, 1.0/temperature)

        probs = probs / np.sum(probs)

        chosen_word = np.random.choice(words, p=probs)
        return chosen_word

    def generate_text(self, seed_text=None, max_length=15, temperature=1.0):
        if seed_text is None:
            if self.n == 1:
                generated = []
            else:
                generated = ['<s>']
        else:
            generated = seed_text.copy()

        for _ in range(max_length):
            next_word = self.sample_next_word(generated, temperature)

            if next_word is None or next_word == '</s>':
                break

            generated.append(next_word)

        if generated and generated[0] == '<s>':
            generated = generated[1:]

        return generated

print("Creating Text Generators...")
unigram_generator = TextGenerator(unigram_model)
bigram_generator = TextGenerator(bigram_model)
trigram_generator = TextGenerator(trigram_model)

def display_generated_text(words, title):
    text = ' '.join(words)
    print(f"\n{title}:")
    print(f"  {text}")
    print(f"  Length: {len(words)} words")

print("\n" + "="*60)
print("TEXT GENERATION EXPERIMENTS")
print("="*60)

print("\n1. GENERATING FROM SCRATCH (No seed text)")
print("-" * 40)

random.seed(42)
np.random.seed(42)

unigram_text = unigram_generator.generate_text(max_length=12, temperature=1.0)
bigram_text = bigram_generator.generate_text(max_length=12, temperature=1.0)
trigram_text = trigram_generator.generate_text(max_length=12, temperature=1.0)

display_generated_text(unigram_text, "Unigram Model")
display_generated_text(bigram_text, "Bigram Model")
display_generated_text(trigram_text, "Trigram Model")

print("\n\n2. CONTINUING HEADLINES (With seed text)")
print("-" * 40)

seed_texts = [
    ['police', 'investigate'],
    ['government', 'announces'],
    ['fire', 'crews']
]

for i, seed in enumerate(seed_texts):
    print(f"\nSeed {i+1}: '{' '.join(seed)}'")

    seed_processed = replace_oov_words(seed, vocabulary)

    unigram_cont = unigram_generator.generate_text(seed_processed.copy(), max_length=8, temperature=1.0)
    bigram_cont = bigram_generator.generate_text(seed_processed.copy(), max_length=8, temperature=1.0)
    trigram_cont = trigram_generator.generate_text(seed_processed.copy(), max_length=8, temperature=1.0)

    display_generated_text(unigram_cont, "  Unigram")
    display_generated_text(bigram_cont, "  Bigram")
    display_generated_text(trigram_cont, "  Trigram")

print("\n\n3. TEMPERATURE EFFECTS (Bigram model with different temperatures)")
print("-" * 40)

seed = ['police', 'arrest']
seed_processed = replace_oov_words(seed, vocabulary)

temperatures = [0.5, 1.0, 1.5]
for temp in temperatures:
    generated = bigram_generator.generate_text(seed_processed.copy(), max_length=10, temperature=temp)
    display_generated_text(generated, f"  Temperature {temp}")

print("\n\n4. DETERMINISTIC vs RANDOM GENERATION")
print("-" * 40)

def generate_deterministic(generator, seed_text, max_length=8):
    if seed_text is None:
        generated = ['<s>'] if generator.n > 1 else []
    else:
        generated = seed_text.copy()

    for _ in range(max_length):
        candidates = generator.get_next_word_candidates(generated)

        if not candidates:
            break

        best_word = max(candidates, key=lambda x: x[1])[0]

        if best_word == '</s>':
            break

        generated.append(best_word)

    if generated and generated[0] == '<s>':
        generated = generated[1:]

    return generated

seed = ['government']
seed_processed = replace_oov_words(seed, vocabulary)

deterministic = generate_deterministic(bigram_generator, seed_processed.copy())
random_sample = bigram_generator.generate_text(seed_processed.copy(), max_length=8, temperature=1.0)

display_generated_text(deterministic, "Deterministic (most probable)")
display_generated_text(random_sample, "Random sampling")

print("\n\n5. PROBABILITY ANALYSIS")
print("-" * 40)

context = ['police']
context_processed = replace_oov_words(context, vocabulary)
candidates = bigram_generator.get_next_word_candidates(context_processed)

print(f"After '{context[0]}', most likely next words:")
sorted_candidates = sorted(candidates, key=lambda x: x[1], reverse=True)[:8]
for word, prob in sorted_candidates:
    print(f"  {word:>12}: {prob:.4f}")

print(f"\n\n6. MULTIPLE SAMPLES (Same seed, different random samples)")
print("-" * 40)

seed = ['fire', 'in']
seed_processed = replace_oov_words(seed, vocabulary)

print(f"Seed: '{' '.join(seed)}'")
print("Multiple bigram generations:")

for i in range(5):
    generated = bigram_generator.generate_text(seed_processed.copy(), max_length=6, temperature=1.0)
    text = ' '.join(generated)
    print(f"  Sample {i+1}: {text}")

print("\nText generation experiments completed!")

Creating Text Generators...

TEXT GENERATION EXPERIMENTS

1. GENERATING FROM SCRATCH (No seed text)
----------------------------------------

Unigram Model:
  will batty leader tax
  Length: 4 words

Bigram Model:
  us
  Length: 1 words

Trigram Model:
  
  Length: 0 words


2. CONTINUING HEADLINES (With seed text)
----------------------------------------

Seed 1: 'police investigate'

  Unigram:
  police investigate dingo fatal flights <s> paddy anderson
  Length: 8 words

  Bigram:
  police investigate
  Length: 2 words

  Trigram:
  police investigate fire at port
  Length: 5 words

Seed 2: 'government announces'

  Unigram:
  government announces to tough
  Length: 4 words

  Bigram:
  government announces review
  Length: 3 words

  Trigram:
  government announces first dam in three days of wait and
  Length: 10 words

Seed 3: 'fire crews'

  Unigram:
  fire crews <s> stanthorpe luxford collapse to <s> plays lead
  Length: 10 words

  Bigram:
  fire crews battle for action vote
  

## 📝 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (50%)**
   - Proper text preprocessing and tokenization
   - Correct N-gram model implementation
   - Working probability calculations
   - Functional vector space models and similarity measures
   - Text generation implementation

2. **Understanding and Analysis (30%)**
   - Correct interpretation of probability results
   - Meaningful analysis of generated text quality
   - Understanding of different N-gram model behaviors
   - Proper handling of OOV words and sentence boundaries

3. **Code Quality and Documentation (20%)**
   - Clean, readable code with proper comments
   - Efficient implementation
   - Good function design and modularity
   - Clear explanations of results